In [ ]:
"""
Author  : A fellow Ravenclaw wizard 🧙‍♂️
Purpose : Benchmark every (preprocessing, model) pair on a regression task
          and visualise the results as a heatmap.
          
How to use
----------
1. Place your tabular dataset in a pandas DataFrame named `df`.
2. Put the target column name in `TARGET_COL`.
3. Edit `preprocessors` and `models` lists to include
   the transformers / estimators you want to compare.
4. Choose `SCORING` = 'r2' or 'neg_mean_absolute_error'.
5. Run the script — a PNG heatmap will be saved next to it.
"""

import itertools
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import get_scorer
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.compose import make_column_selector as selector

# ───────────────────────────────────────────────────────
# 📜 1. USER CONFIGURATION
# ───────────────────────────────────────────────────────
TARGET_COL = "target"        # ← replace with your own target column
SCORING    = "r2"            # or "neg_mean_absolute_error"
N_SPLITS   = 5               # CV folds

# Example dataset — replace with your own ------------------------------------------------
# df = pd.read_csv("your_dataset.csv")
# For demo purposes, we conjure up a dummy set.
from sklearn.datasets import fetch_california_housing
cal = fetch_california_housing(as_frame=True)
df  = cal.frame.copy()
TARGET_COL = "MedHouseVal"

# ───────────────────────────────────────────────────────
# 🧩 2. DEFINE PREPROCESSORS
# Each entry must be a (name, transformer) pair.
# ───────────────────────────────────────────────────────
numeric_selector = selector(dtype_include=np.number)
categorical_selector = selector(dtype_exclude=np.number)

basic_numeric = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

basic_categorical = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine column‑wise
default_preprocessor = ColumnTransformer([
    ("num", basic_numeric, numeric_selector),
    ("cat", basic_categorical, categorical_selector),
])

preprocessors = [
    ("identity", "passthrough"),          # no preprocessing
    ("default",  default_preprocessor),
]

# ───────────────────────────────────────────────────────
# 🦾 3. DEFINE MODELS
# Each entry must be a (name, estimator) pair.
# ───────────────────────────────────────────────────────
models = [
    ("Ridge",  Ridge()),
    ("Lasso",  Lasso(max_iter=10_000)),
    ("RF",     RandomForestRegressor(n_estimators=200, random_state=42)),
    ("GBR",    GradientBoostingRegressor(random_state=42)),
]

# ───────────────────────────────────────────────────────
# 🔮 4. BENCHMARK ALL COMBINATIONS
# ───────────────────────────────────────────────────────
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

results = []  # rows: (preproc_name, model_name, score)
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
scorer = get_scorer(SCORING)

for (pp_name, pp) in preprocessors:
    for (mdl_name, mdl) in models:
        pipe = Pipeline([
            ("prep", pp),
            ("model", mdl)
        ])
        cv_scores = cross_val_score(pipe, X, y,
                                    cv=kf, scoring=scorer)
        mean_score = cv_scores.mean()
        results.append((pp_name, mdl_name, mean_score))
        print(f"{pp_name} + {mdl_name} → {mean_score:.4f}")

# Convert to DataFrame
scores_df = pd.DataFrame(results, columns=["Preprocessing", "Model", "Score"])

# Pivot for heatmap
heatmap_df = scores_df.pivot(index="Model", columns="Preprocessing", values="Score")

# ───────────────────────────────────────────────────────
# 🎨 5. PLOT HEATMAP
# ───────────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
sns.heatmap(
    heatmap_df,
    annot=True,
    fmt=".3f",
    linewidths=0.5,
    cmap="viridis",
    cbar_kws={"label": SCORING.upper()}
)
plt.title(f"Model × Preprocessing performance ({SCORING})")
plt.tight_layout()
plt.savefig("model_preprocessing_heatmap.png", dpi=300)
plt.show()
